# MNIST Handwritten Digit Recognition with a CNN

Trains a Convolutional Neural Network (CNN) to recognize handwritten digits 0-9 from the MNIST dataset
(60,000 training and 10,000 test images, each 28 x 28 grayscale pixels).

**Workflow:** load data -> explore -> preprocess -> build CNN -> train -> evaluate -> visualize predictions

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report

# Reproducibility: the same seed gives (almost) the same results on each run
tf.keras.utils.set_random_seed(42)

## 2. Load the data and save it as CSV

The original dataset link was unavailable, so the data is loaded from Keras and written to CSV files
(`label` + 784 pixel columns named `1x1` ... `28x28`). The CSV step only runs if the files do not exist yet.

In [ ]:
CSV_TRAIN, CSV_TEST = "mnist_train.csv", "mnist_test.csv"

if not (os.path.exists(CSV_TRAIN) and os.path.exists(CSV_TEST)):
    (X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = mnist.load_data()

    # Column names: row x column of each pixel, e.g. "1x1", "1x2", ... "28x28"
    pixel_cols = [f"{r}x{c}" for r in range(1, 29) for c in range(1, 29)]

    for X, y, path in [(X_train_raw, y_train_raw, CSV_TRAIN),
                       (X_test_raw, y_test_raw, CSV_TEST)]:
        frame = pd.DataFrame(X.reshape(-1, 784), columns=pixel_cols)
        frame.insert(0, "label", y)          # label goes in the first column
        frame.to_csv(path, index=False)

df = pd.read_csv(CSV_TRAIN)
df_test = pd.read_csv(CSV_TEST)

## 3. Explore the data

In [ ]:
print("Train shape:", df.shape)       # expected: (60000, 785) -> 1 label + 784 pixels
print("Test shape: ", df_test.shape)  # expected: (10000, 785)
print("Missing values:", df.isnull().sum().sum())
df.head()

In [ ]:
# Class balance: how many images of each digit?
df["label"].value_counts().sort_index().plot(kind="bar", figsize=(8, 3), title="Images per digit (train)")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.show()

## 4. Preprocessing

1. Split labels (`y`) from pixels (`x`)
2. Scale pixels from 0-255 to 0-1 (helps training converge faster)
3. Reshape each row of 784 values into a 28 x 28 x 1 image (the last 1 is the grayscale channel)
4. One-hot encode the labels (e.g. 5 -> `[0,0,0,0,0,1,0,0,0,0]`)

In [ ]:
# 1. Split features and labels
x_train = df.drop("label", axis=1).values
y_train = df["label"].values
x_test = df_test.drop("label", axis=1).values
y_test = df_test["label"].values

# 2. Normalize pixel values to the 0-1 range
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# 3. Reshape to (samples, height, width, channels), the format a CNN expects
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# 4. One-hot encode labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print("Train images:", x_train.shape, "| Train labels:", y_train_cat.shape)

In [ ]:
# Sanity check: show the first 10 training images with their labels
plt.figure(figsize=(12, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train[i].reshape(28, 28), cmap="gray")
    plt.title(y_train[i])
    plt.axis("off")
plt.show()

## 5. Build the CNN

| Layer | Purpose |
|---|---|
| `Conv2D(32)` + `MaxPooling2D` | detect simple features (edges, strokes) and shrink the image |
| `Conv2D(64)` + `MaxPooling2D` | combine them into larger shapes (loops, curves) |
| `Flatten` | turn the feature maps into one long vector |
| `Dense(128)` | learn which feature combinations match which digit |
| `Dropout(0.5)` | randomly switch off half the units during training to reduce overfitting |
| `Dense(10, softmax)` | output one probability for each digit 0-9 |

In [ ]:
model = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(10, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",   # matches the one-hot encoded labels
    metrics=["accuracy"],
)
model.summary()

## 6. Train

10% of the training data is held out as a validation set so we can watch for overfitting.
The test set is kept untouched until the final evaluation. `EarlyStopping` stops training if the
validation loss has not improved for 3 epochs and restores the best weights.

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history = model.fit(
    x_train, y_train_cat,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1,
)

## 7. Training curves

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.title("CNN Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.title("CNN Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

## 8. Evaluate on the test set

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f"Test accuracy: {test_acc:.4f}  ({test_acc * 100:.2f}%)")
print(f"Test loss:     {test_loss:.4f}")

In [ ]:
# Predicted digit for every test image
y_pred = model.predict(x_test, verbose=0).argmax(axis=1)

# Precision, recall and F1-score for each digit
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
# Confusion matrix: rows = true digit, columns = predicted digit
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.title("Confusion Matrix")
plt.xlabel("Predicted digit")
plt.ylabel("True digit")
plt.xticks(range(10))
plt.yticks(range(10))
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
plt.show()

## 9. Visualize predictions

In [ ]:
# Random test images with the true label and the model's prediction
idxs = np.random.choice(len(x_test), 10, replace=False)

plt.figure(figsize=(15, 3))
for i, idx in enumerate(idxs):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_test[idx].reshape(28, 28), cmap="gray")
    plt.axis("off")
    correct = y_pred[idx] == y_test[idx]
    plt.title(f"True: {y_test[idx]}\nPred: {y_pred[idx]}", color="green" if correct else "red")
plt.tight_layout()
plt.show()

In [ ]:
# Look at images the model got wrong (useful for understanding its mistakes)
wrong = np.where(y_pred != y_test)[0]
print(f"Misclassified: {len(wrong)} of {len(y_test)} test images")

plt.figure(figsize=(15, 3))
for i, idx in enumerate(wrong[:10]):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_test[idx].reshape(28, 28), cmap="gray")
    plt.axis("off")
    plt.title(f"True: {y_test[idx]}\nPred: {y_pred[idx]}", color="red")
plt.tight_layout()
plt.show()

## 10. Save the trained model

In [ ]:
model.save("mnist_cnn_model.keras")
print("Model saved to mnist_cnn_model.keras")

## Summary
A small CNN with two convolution blocks reaches about **99% test accuracy** on MNIST.
Convolution layers work well here because they use the 2D structure of the image, which a plain
fully connected network ignores. Check the exact accuracy printed in section 8.